In [21]:
from pathlib import Path

import pandas as pd

MAX_REASONING_CHARS = 24000
DEFAULT_TAIL_CHARS = 6000  # tail kept when corrected_reasoning is empty (head 18000 + tail 6000)

ORIGINAL_TRAIN_PATH = Path("../../data/out/splits/random/mmlu/train_original.parquet")
DISTILL_PATH = Path("../../data/out/distillation/mmlu_corrected_answer_deepseek_v4_pro_and_others.parquet")
OUT_DIR = Path("../../data/out/splits/random/mmlu/")
OUT_NAME_HEAD = f"train_corrected_answer_deepseek_v4_pro_and_others_head_truncated{MAX_REASONING_CHARS}.parquet"
OUT_NAME_MIDDLE = f"train_corrected_answer_deepseek_v4_pro_and_others_middle_truncated{MAX_REASONING_CHARS}.parquet"
OUT_NAME_TAIL = f"train_corrected_answer_deepseek_v4_pro_and_others_tail_truncated{MAX_REASONING_CHARS}.parquet"


In [22]:
train_ids = set(pd.read_parquet(ORIGINAL_TRAIN_PATH)["question_id"])
distill_df = pd.read_parquet(DISTILL_PATH)
print(f"Distill rows: {len(distill_df)}, original train ids: {len(train_ids)}")

train_df = distill_df[distill_df["question_id"].isin(train_ids)].reset_index(drop=True)
assert len(train_df) == len(train_ids), f"Expected {len(train_ids)} rows, got {len(train_df)}"
print(f"Filtered train rows: {len(train_df)}")

Distill rows: 12032, original train ids: 9626
Filtered train rows: 9626


In [23]:
KEEP_ONLY_CORRECT = False

if KEEP_ONLY_CORRECT:
    before = len(train_df)
    train_df = train_df[train_df["distill_ans_correct"]].reset_index(drop=True)
    print(f"Kept only correct: {before} -> {len(train_df)} rows")
else:
    print(f"Keeping all rows ({len(train_df)})")

Keeping all rows (9626)


In [24]:
def middle_truncate(
    reasoning,
    corrected,
    max_chars=MAX_REASONING_CHARS,
    default_tail=DEFAULT_TAIL_CHARS,
):
    """Drop the middle of `reasoning`, keeping a head + tail of `max_chars` total.

    Tail length equals len(corrected_reasoning) when it is non-empty, otherwise
    `default_tail`. NaN / already-short reasoning is returned unchanged.
    """
    if not isinstance(reasoning, str) or len(reasoning) <= max_chars:
        return reasoning
    has_corrected = isinstance(corrected, str) and corrected.strip() != ""
    tail = min(len(corrected), max_chars) if has_corrected else default_tail
    head = max_chars - tail
    if head <= 0:  # corrected_reasoning >= budget -> keep last max_chars
        return reasoning[-max_chars:]
    return reasoning[:head] + reasoning[-tail:]

In [25]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

too_long = int((train_df["distill_reasoning"].str.len() > MAX_REASONING_CHARS).sum())

# Variant 1: head truncation (first MAX_REASONING_CHARS chars)
head_df = train_df.copy()
head_df["distill_reasoning"] = head_df["distill_reasoning"].str.slice(0, MAX_REASONING_CHARS)
head_df.to_parquet(str(OUT_DIR / OUT_NAME_HEAD), index=False)

# Variant 3: tail truncation (last DEFAULT_TAIL_CHARS chars)
tail_df = train_df.copy()
tail_df["distill_reasoning"] = tail_df["distill_reasoning"].str.slice(-DEFAULT_TAIL_CHARS, None)
tail_df.to_parquet(str(OUT_DIR / OUT_NAME_TAIL), index=False)

# Variant 2: middle drop (head + tail, tail sized by corrected_reasoning).
# If corrected_reasoning is absent, fall back to empty -> head 18000 + tail 6000.
middle_df = train_df.copy()
corrected = (
    middle_df["corrected_reasoning"]
    if "corrected_reasoning" in middle_df.columns
    else pd.Series([None] * len(middle_df), index=middle_df.index)
)
middle_df["distill_reasoning"] = [middle_truncate(r, c) for r, c in zip(middle_df["distill_reasoning"], corrected)]
middle_df.to_parquet(str(OUT_DIR / OUT_NAME_MIDDLE), index=False)

print(f"Truncated distill_reasoning in {too_long} rows (budget {MAX_REASONING_CHARS} chars)")
print(f"Saved head variant   -> {(OUT_DIR / OUT_NAME_HEAD).resolve()}")
print(f"Saved middle variant -> {(OUT_DIR / OUT_NAME_MIDDLE).resolve()}")
print(f"Saved tail variant   -> {(OUT_DIR / OUT_NAME_TAIL).resolve()}")

Truncated distill_reasoning in 749 rows (budget 24000 chars)
Saved head variant   -> /Users/aigoncharov/dev/sktech/recursive_caft/data/out/splits/random/mmlu/train_corrected_answer_deepseek_v4_pro_and_others_head_truncated24000.parquet
Saved middle variant -> /Users/aigoncharov/dev/sktech/recursive_caft/data/out/splits/random/mmlu/train_corrected_answer_deepseek_v4_pro_and_others_middle_truncated24000.parquet
Saved tail variant   -> /Users/aigoncharov/dev/sktech/recursive_caft/data/out/splits/random/mmlu/train_corrected_answer_deepseek_v4_pro_and_others_tail_truncated24000.parquet
